# Financial Trading Pipeline: Demonstration

## Section 1: Introduction and Setup

This notebook demonstrates the capabilities of the `financial_pipeline` library. We will walk through:
- Data Ingestion
- Feature Engineering (basic and custom transformers)
- Model Training and Cross-Validation
- Using Custom Metrics
- Hyperparameter Optimization with Optuna
- Saving and Loading Objects (models, studies)

The goal is to provide a hands-on guide to using the various components of the pipeline for financial machine learning tasks.

In [ ]:
# Common libraries
import pandas as pd
import numpy as np
import logging
import os
from pathlib import Path

# Scikit-learn
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score
from sklearn.base import clone

# Optuna (for optimization section)
import optuna

# Financial Pipeline components
# Assuming the notebook is in `financial_pipeline/notebooks/` and the pipeline modules are in the parent directory.
# Add parent directory to sys.path to allow direct imports if not installed as a package.
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from data_ingestion import fetch_financial_data
from feature_engineering.base import BaseFeatureTransformer # For custom transformer
from feature_engineering import MovingAverageTransformer # Example built-in
from models import LGBMWrapper, SimpleNNWrapper # Example models
from cross_validation import BasicTimeSeriesSplit, WalkForwardSplit # Example CV splitters
from evaluation import metrics as eval_metrics # Contains re-exported sklearn metrics and custom ones
from evaluation.plotting import setup_tensorboard_writer, log_metrics_to_tensorboard, plot_reliability_diagram
from optimization import run_optimization # For HPO
from utils.logging_utils import setup_basic_logger
from utils.io_utils import save_object, load_object

# Setup basic logger for the pipeline
logger = setup_basic_logger(logger_name='FinancialPipelineDemo', log_level=logging.INFO)
logger.info("Setup complete. Necessary modules imported and logger configured.")

## Section 2: Data Ingestion

We'll start by fetching some financial data using the `fetch_financial_data` utility, which internally uses `yfinance`.

In [ ]:
data_fetcher_params = {
    'tickers': 'GOOGL', 
    'start_date': '2022-01-01',
    'end_date': '2023-01-01',
    'interval': '1d'
}

raw_df = fetch_financial_data(**data_fetcher_params)

if not raw_df.empty:
    logger.info(f"Fetched data for {data_fetcher_params['tickers']}:")
    print("--- Data Head ---")
    print(raw_df.head())
    print("\n--- Data Info ---")
    raw_df.info()
else:
    logger.warning("Failed to fetch data. Subsequent steps might fail.")

### Creating a Target Variable

For supervised learning, we need a target variable. A common task is to predict the next period's price or return. Here's how you might create a target representing the next day's closing price:

In [ ]:
if not raw_df.empty:
    df_with_target = raw_df.copy()
    df_with_target['Target'] = df_with_target['Close'].shift(-1)
    
    # Drop the last row since it will have a NaN target
    original_len = len(df_with_target)
    df_with_target.dropna(subset=['Target'], inplace=True)
    logger.info(f"Target created. Dropped {original_len - len(df_with_target)} row(s) with NaN target.")
    
    print("\n--- DataFrame with Target (Tail) ---")
    print(df_with_target.tail())
    
    # Separate features (X) and target (y) for further processing
    # X should not include the target column itself or any future information leaked from it.
    X_full = df_with_target.drop(columns=['Target'])
    y_full = df_with_target['Target']
    
    logger.info(f"X_full shape: {X_full.shape}, y_full shape: {y_full.shape}")
else:
    logger.warning("Skipping target creation as raw_df is empty.")
    # Create empty placeholders if needed for later cells not to error out immediately
    X_full = pd.DataFrame()
    y_full = pd.Series(dtype='float64')

## Section 3: Feature Engineering - Basic

Our pipeline includes feature transformers that can be used within an `sklearn.pipeline.Pipeline`. Let's demonstrate with the `MovingAverageTransformer`.

In [ ]:
if not X_full.empty:
    sma_transformer = MovingAverageTransformer(window_sizes=[10, 20], column='Close')
    
    # Create a scikit-learn pipeline for features
    feature_pipeline_basic = Pipeline([
        ('sma_features', sma_transformer)
    ])
    
    # Apply to a portion of data (X_full already excludes the target)
    # Ensure y_full (target) is not passed to fit/transform of feature engineering pipeline
    # if features are purely based on X.
    # Some transformers might need y (e.g., for target encoding), sklearn pipelines handle this.
    X_example_transformed_basic = feature_pipeline_basic.fit_transform(X_full.copy()) 
    # For transformers that might add NaNs at the beginning (like SMAs):
    X_example_transformed_basic.dropna(inplace=True) 
    
    logger.info("Applied basic feature engineering pipeline.")
    print("--- Transformed Data with Basic Features (Head) ---")
    print(X_example_transformed_basic.head())
    print("\n--- Transformed Data with Basic Features (Info) ---")
    X_example_transformed_basic.info()
else:
    logger.warning("Skipping basic feature engineering as X_full is empty.")

## Section 4: Feature Engineering - Custom Transformer Example

You can easily create and integrate your own custom feature transformers by inheriting from `BaseFeatureTransformer` (which itself inherits from scikit-learn's `BaseEstimator` and `TransformerMixin`).

In [ ]:
# Define a simple custom transformer
class LagFeatureTransformer(BaseFeatureTransformer):
    """Adds lagged versions of a specified column."""
    def __init__(self, column_to_lag: str, lag_periods: list):
        super().__init__()
        if not isinstance(lag_periods, list) or not all(isinstance(p, int) and p > 0 for p in lag_periods):
            raise ValueError("lag_periods must be a list of positive integers.")
        self.column_to_lag = column_to_lag
        self.lag_periods = lag_periods
        self.new_columns_ = []

    def fit(self, X: pd.DataFrame, y=None):
        if self.column_to_lag not in X.columns:
            raise ValueError(f"Column '{self.column_to_lag}' not found in input DataFrame.")
        self.new_columns_ = [f'{self.column_to_lag}_lag_{p}' for p in self.lag_periods]
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X_transformed = X.copy()
        if self.column_to_lag not in X_transformed.columns:
            raise ValueError(f"Column '{self.column_to_lag}' not found during transform.")
            
        for period in self.lag_periods:
            col_name = f'{self.column_to_lag}_lag_{period}'
            X_transformed[col_name] = X_transformed[self.column_to_lag].shift(period)
        
        # Fill NaNs introduced by lagging (can use the helper from BaseFeatureTransformer if complex)
        # For simplicity here, just basic ffill/bfill on new columns
        for col in self.new_columns_:
            if col in X_transformed.columns: # Check because fit might not have run on this exact X instance
                 X_transformed[col] = X_transformed[col].fillna(method='ffill').fillna(method='bfill')
        return X_transformed

if not X_full.empty:
    # Instantiate the custom transformer
    custom_lag_transformer = LagFeatureTransformer(column_to_lag='Close', lag_periods=[1, 2, 3])
    
    # Create a new feature pipeline including it
    # Assuming sma_transformer was defined in the previous cell
    feature_pipeline_custom = Pipeline([
        ('sma_features', MovingAverageTransformer(window_sizes=[5], column='Close')), # Example SMA
        ('lag_features', custom_lag_transformer)
    ])
    
    X_example_transformed_custom = feature_pipeline_custom.fit_transform(X_full.copy())
    X_example_transformed_custom.dropna(inplace=True) # Handle NaNs from SMAs and Lags

    logger.info("Applied custom feature engineering pipeline.")
    print("--- Transformed Data with Custom Features (Head) ---")
    print(X_example_transformed_custom.head())
else:
    logger.warning("Skipping custom feature engineering as X_full is empty.")

## Section 5: Model Training & Cross-Validation

Now, let's use the `train_and_evaluate_model` function to train a model and evaluate it using time series cross-validation. We'll use the basic feature pipeline for this example.

In [ ]:
if not X_full.empty and not y_full.empty:
    # Model (LGBM Regressor)
    lgbm_model_wrapper = LGBMWrapper(
        objective='regression',
        metric='rmse', 
        n_estimators=50, # Small for demo
        learning_rate=0.05,
        random_state=42,
        n_jobs=1
    )
    
    # CV Splitter
    # Using fewer splits and smaller test_size for faster demo execution
    cv_splitter_main = BasicTimeSeriesSplit(n_splits=3, test_size=30) 
    
    # Target column name (must match the one created earlier)
    target_col = 'Target'
    
    # Metrics to calculate
    # The functions themselves are passed, not their string names.
    metrics_dict_main = [eval_metrics.mean_squared_error, eval_metrics.mean_absolute_error]
    
    # TensorBoard log directory for this CV run
    tb_log_dir_cv = os.path.join('logs', 'notebook_cv_run', pd.Timestamp.now().strftime('%Y%m%d_%H%M%S'))
    logger.info(f"TensorBoard logs for CV will be saved to: {tb_log_dir_cv}")

    # Ensure the feature pipeline used here is the one we want (e.g., basic or custom)
    # Let's use feature_pipeline_basic which was defined with SMAs.
    # Important: train_and_evaluate_model expects an unfitted feature pipeline.
    # We re-instantiate it here to be sure.
    current_fe_pipeline = Pipeline([
        ('sma_features', MovingAverageTransformer(window_sizes=[10, 20], column='Close'))
    ])

    logger.info("Starting model training and cross-validation...")
    trained_models, train_metrics, val_metrics, test_metrics = train_and_evaluate_model(
        data_fetcher_params=data_fetcher_params, # Defined in Section 2
        feature_engineering_pipeline=current_fe_pipeline,
        model_wrapper=lgbm_model_wrapper,
        cv_splitter=cv_splitter_main,
        target_column_name=target_col, # Make sure this matches the target created in df_with_target
        metrics_to_calculate=metrics_dict_main,
        walk_forward_val_test_scheme=False, # BasicTimeSeriesSplit yields (train, test)
        tensorboard_log_dir=tb_log_dir_cv
    )
    
    logger.info("Finished model training and cross-validation.")
    if test_metrics:
        print("\n--- CV Test Metrics (per fold) ---")
        for i, fold_metric in enumerate(test_metrics):
            print(f"Fold {i+1}: {fold_metric}")
        
        # Calculate and print average test metrics
        avg_test_results = {}
        for key in test_metrics[0].keys(): # Assumes all folds have same metric keys
            avg_test_results[key] = np.mean([m[key] for m in test_metrics])
        print("\n--- Average CV Test Metrics ---")
        print(avg_test_results)
        logger.info(f"Average CV Test Metrics: {avg_test_results}")
    else:
        logger.warning("No test metrics were returned from CV.")
        
    logger.info(f"TensorBoard logs for this run can be viewed with: tensorboard --logdir {os.path.abspath('logs/notebook_cv_run')}")
else:
    logger.warning("Skipping model training & CV as data is not loaded/prepared.")

## Section 6: Custom Metric Example

You can define your own custom metric functions and pass them to `train_and_evaluate_model`. The function should typically take `(y_true, y_pred)` as arguments and return a scalar value.

In [ ]:
# Define a simple custom metric: Sign Agreement
# This metric checks if the predicted change direction matches the actual change direction.
# For this to be meaningful, y_true and y_pred should represent changes (e.g., returns or price differences).
# If using with 'Target' as next day's price, this metric might not be directly interpretable unless 
# y_true and y_pred are transformed into changes first (e.g., y_pred - current_price).

def sign_agreement_metric(y_true, y_pred):
    """Calculates sign agreement between true and predicted values.
       More meaningful if y_true and y_pred are changes/returns.
    """
    # Example: If predicting price, convert to change from a reference (e.g., previous day's price)
    # This requires passing more info or assuming y_true/y_pred are already price changes.
    # For simplicity, let's assume they are already comparable in terms of sign.
    return np.mean(np.sign(y_true) == np.sign(y_pred))

logger.info(f"Custom metric 'sign_agreement_metric' defined.")

# How to use it:
metrics_with_custom = [
    eval_metrics.mean_squared_error, 
    eval_metrics.mean_absolute_error,
    sign_agreement_metric # Add the custom function
]

logger.info("To use this, pass `metrics_with_custom` to `train_and_evaluate_model`'s `metrics_to_calculate` argument.")
print("Updated list of metrics to calculate would be:", [m.__name__ for m in metrics_with_custom])

# You would then re-run train_and_evaluate_model with this updated list.
# For brevity, we won't re-run the full CV here but just show the setup.

## Section 7: Hyperparameter Optimization with Optuna

The pipeline supports hyperparameter optimization using Optuna. We need to define search spaces for feature engineering parameters and model parameters.

In [ ]:
if not X_full.empty and not y_full.empty:
    logger.info("Setting up Optuna hyperparameter optimization...")
    
    # 1. Define Feature Engineering Search Space
    # Using the custom LagFeatureTransformer defined earlier for this example.
    fe_search_space_optuna = [
        {
            'class': MovingAverageTransformer,
            'params': {
                'window_sizes': ('suggest_categorical', 'sma_window_choices', [[5], [10, 20], [5,15,25]]),
                'column': ('suggest_categorical', 'sma_col_choice', ['Close', 'Open'])
            }
        },
        {
            'class': LagFeatureTransformer, # Our custom transformer
            'params': {
                'column_to_lag': ('suggest_categorical', 'lag_col_choice', ['Close', 'Volume']),
                # Example: suggest a list of 3 lags, each between 1 and 5
                # This is a bit complex for Optuna's direct suggestion. A simpler way is to categorize choices:
                'lag_periods': ('suggest_categorical', 'lag_period_choices', [[1,2,3], [1,3,5], [2,4]])
            }
        }
    ]
    
    # 2. Define Model Configuration Search Space (for LGBMWrapper)
    model_search_space_optuna = {
        'class': LGBMWrapper,
        'static_params': { # Static parameters for the model
            'objective': 'regression',
            'metric': 'rmse', # Internal LGBM metric
            'random_state': 42,
            'n_jobs': 1
        },
        'params': { # Parameters to be tuned by Optuna
            'n_estimators': ('suggest_int', 'lgbm_n_est', 30, 100), # Smaller range for demo
            'learning_rate': ('suggest_float', 'lgbm_lr', 0.01, 0.1, {'log': True}),
            'num_leaves': ('suggest_int', 'lgbm_num_leaves', 10, 30)
        }
    }
    
    # 3. CV Splitter Configuration (static for optimization)
    cv_config_optuna = {
        'class': BasicTimeSeriesSplit,
        'params': {'n_splits': 2, 'test_size': 20} # Very few splits/small test for faster demo
    }
    
    # 4. Metric to Optimize and Direction
    opt_metric_func_optuna = eval_metrics.mean_squared_error
    opt_metric_higher_is_better_optuna = False # Lower MSE is better
    
    # 5. TensorBoard Log Directory for the Optuna Study
    # Each trial will get its own sub-directory under this.
    optuna_study_tb_log_prefix = os.path.join('logs', 'notebook_optuna_study') 
    
    # 6. Run Optimization
    logger.info(f"Starting Optuna study. Number of trials: 3 (DEMO). Logging to {optuna_study_tb_log_prefix}")
    try:
        study = run_optimization(
            data_fetcher_params=data_fetcher_params, # Defined in Section 2
            feature_engineering_config_search_space=fe_search_space_optuna,
            model_config_search_space=model_search_space_optuna,
            cv_splitter_config=cv_config_optuna,
            target_column_name=target_col, # 'Target'
            metric_to_optimize=opt_metric_func_optuna,
            metric_greater_is_better=opt_metric_higher_is_better_optuna,
            walk_forward_val_test_scheme=False, # BasicTimeSeriesSplit used
            n_trials=3,  # VERY small number for quick demonstration
            study_name='notebook_financial_opt_demo',
            storage_url=None, # In-memory for this demo
            tensorboard_log_dir_study_prefix=optuna_study_tb_log_prefix
        )
        
        logger.info("Optuna study finished.")
        print("\n--- Optuna Best Trial ---")
        print(f"Value (Optimized Metric): {study.best_trial.value}")
        print("Params:")
        for key, value in study.best_trial.params.items():
            print(f"  {key}: {value}")
    except Exception as e:
        logger.error(f"An error occurred during Optuna optimization: {e}", exc_info=True)
else:
    logger.warning("Skipping Optuna optimization as data is not loaded/prepared.")

## Section 8: Saving and Loading Objects

The `utils.io_utils` module provides helper functions `save_object` and `load_object` (using `joblib`) to easily persist and retrieve Python objects like trained models, feature pipelines, or Optuna study objects.

In [ ]:
logger.info("Demonstrating saving and loading objects...")

# Example: Save the Optuna study object (if it exists from previous cell)
optuna_study_filepath = Path('temp_optuna_study.joblib')
if 'study' in locals() and study is not None: # Check if 'study' was created
    try:
        save_object(study, optuna_study_filepath)
        logger.info(f"Optuna study object saved to {optuna_study_filepath}")
        
        loaded_study = load_object(optuna_study_filepath)
        logger.info(f"Optuna study object loaded successfully. Type: {type(loaded_study)}")
        if loaded_study:
             print(f"Loaded study best trial value: {loaded_study.best_trial.value}")
    except Exception as e:
        logger.error(f"Error during Optuna study save/load demo: {e}", exc_info=True)
    finally:
        if optuna_study_filepath.exists():
            optuna_study_filepath.unlink() # Clean up
else:
    logger.warning("Optuna 'study' object not found, skipping save/load demonstration for it.")

# Example: Save one of the trained models from the CV step (if it exists)
trained_model_filepath = Path('temp_trained_model.joblib')
if 'trained_models' in locals() and trained_models:
    try:
        model_to_save = trained_models[0] # Save the model from the first fold
        save_object(model_to_save, trained_model_filepath)
        logger.info(f"Trained model saved to {trained_model_filepath}")
        
        loaded_model = load_object(trained_model_filepath)
        logger.info(f"Trained model loaded successfully. Type: {type(loaded_model)}")
        if hasattr(loaded_model, 'predict') and not X_full.empty:
            # To make a prediction, we'd need appropriately transformed data.
            # This is just a basic check that the model object is loaded.
            logger.info("Loaded model has a 'predict' method.")
    except Exception as e:
        logger.error(f"Error during trained model save/load demo: {e}", exc_info=True)
    finally:
        if trained_model_filepath.exists():
            trained_model_filepath.unlink() # Clean up
else:
    logger.warning("List 'trained_models' not found or empty, skipping save/load demonstration for it.")

## Section 9: Conclusion

This notebook has demonstrated the key functionalities of the `financial_pipeline` library.

Key takeaways:
- **Modular Design**: Components for data ingestion, feature engineering, modeling, cross-validation, evaluation, and optimization are separated for clarity and reusability.
- **Extensibility**: Custom transformers, models (via wrappers), metrics, and CV strategies can be easily integrated.
- **Automation**: Repetitive tasks like cross-validation and hyperparameter tuning are streamlined.
- **Reproducibility**: Utilities for saving/loading objects and structured logging (including TensorBoard) aid in tracking experiments.

You are encouraged to adapt and extend this pipeline for your specific financial machine learning projects. Explore different feature combinations, model architectures, and optimization strategies to build robust trading models.